# Module 11: Temperature Check

## Mission Briefing

In Module 10, your Pico learned to sense **light**. Today, it learns to sense **temperature**.

You'll wire up a **thermistor** — a tiny component that changes how much electricity it lets through based on how hot or cold it is. Same voltage divider trick. Same ADC reading. But this time, the stimulus is heat instead of light.

```
                  COLD                              WARM
         +------------------+               +------------------+
         |   Thermistor     |               |   Thermistor     |
         |   HIGH resistance|               |   LOW resistance |
         +--------+---------+               +--------+---------+
                  |                                   |
                  v                                   v
           Low ADC reading                     High ADC reading
                  |                                   |
                  v                                   v
         +------------------+               +------------------+
         |   Convert to     |               |   Convert to     |
         |   temperature!   |               |   temperature!   |
         |    18.2 C        |               |    31.7 C        |
         +------------------+               +------------------+
```

By the end of today, you'll have a working **digital thermometer** — and even a simple **data logger** that records temperature over time.

---
## Science Spotlight: Temperature-Sensitive Materials

Some materials change how much electricity they let through based on temperature. Just like the photoresistor — but it reacts to heat instead of light. Your instructor will explain how thermistors work and how we convert a raw number into actual degrees.

---
## Meet the Thermistor

A thermistor looks like a small bead or disc with two wire legs. The name is a mashup of **therm**al + res**istor**.

```
           +-------+
           | BEAD  |    <-- temperature-sensitive material
           +---+---+
               |   |
              Pin  Pin
           (no polarity — either direction works)
```

Our thermistor is an **NTC** type (Negative Temperature Coefficient):
- **Cold** = HIGH resistance (electricity has a hard time flowing)
- **Hot** = LOW resistance (electricity flows more easily)

Notice the pattern: as temperature goes UP, resistance goes DOWN. That's the "negative" part.

It has **no polarity** — plug it in either direction.

**Compare to Module 10:** The photoresistor changed with light. The thermistor changes with temperature. Same idea, different trigger!

---
## Wiring: Thermistor Voltage Divider

This circuit looks almost identical to Module 10 — just swap the photoresistor for the thermistor!

### Circuit Diagram

```
   Temperature Sensor (Voltage Divider):

   Pico 3.3V ──── wire ──── Thermistor Pin 1
                             Thermistor Pin 2 ──── junction ──── wire ──── Pico GP26 (ADC0)
                                                      |
                                                 [10k ohm Resistor]
                                                      |
                                                   Pico GND
```

### Step-by-Step

1. **Place the thermistor** on the breadboard — each of its 2 pins in a different row
2. **Connect a wire** from **3.3V** on the Pico to one pin of the thermistor
3. **Connect a wire** from the **other pin** of the thermistor to **GP26** on the Pico
4. **Place the 10k ohm resistor** with one leg in the **same row** as the thermistor/GP26 junction, and the other leg in a different row
5. **Connect a wire** from the other leg of the 10k ohm resistor to **GND**

Plug in your USB cable.

**Note:** If you still have your Module 10 circuit, you can simply swap the photoresistor for the thermistor — everything else stays the same! (Remove the LED circuit for now — we don't need it until the challenge.)

---
## Code Along: Read the Raw Temperature Value

Let's start by seeing what raw numbers the thermistor gives us — just like we did with the photoresistor.

Fill in the blank:

In [ ]:
from machine import ADC
import time

adc = ADC(_____)                # Which GPIO pin is our voltage divider junction on?

for i in range(20):
    raw_value = adc.read_u16()  # Read a 16-bit value (0 to 65535)
    print("Raw ADC value:", raw_value)
    time.sleep(0.5)

Run the code and try these:

- What value do you see at **room temperature**? ____________
- **Hold the thermistor between your fingers** for 10 seconds — what happens? ____________
- **Let go** — does the value slowly return? ____________

You can see the thermistor responding to your body heat! But a raw number like 34521 doesn't tell us the actual temperature. We need to **convert** it.

---
## Code Along: Convert to Real Temperature

Converting a raw ADC reading to actual degrees takes a few math steps. Don't worry if the math looks complicated — your instructor will walk you through it.

Here's the conversion process:
1. Turn the ADC reading into a **resistance** value
2. Turn the resistance into **temperature in Kelvin** (using a special formula)
3. Convert Kelvin to **Celsius** and **Fahrenheit**

Fill in the blanks:

In [ ]:
from machine import ADC
import math
import time

adc = ADC(26)

for i in range(10):
    adc_value = adc.read_u16()
    
    # Step 1: Calculate the thermistor's resistance
    # The voltage divider formula, rearranged:
    resistance = 10000 * adc_value / (_____ - adc_value)   # What's the max ADC value?
    
    # Step 2: Convert resistance to temperature (Steinhart-Hart equation)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance/10000))
    
    # Step 3: Convert Kelvin to Celsius and Fahrenheit
    temp_c = temp_k - _____        # What do you subtract to go from Kelvin to Celsius?
    temp_f = temp_c * 9/5 + 32
    
    print("Temp:", round(temp_c, 1), "C /", round(temp_f, 1), "F")
    time.sleep(1)

Does the temperature look reasonable? Room temperature is usually around **20-25 C** (68-77 F).

Try holding the thermistor between your fingers again — you should see the temperature rise toward **30-35 C** (body heat warms it up)!

---
## Code Along: Continuous Temperature Monitor

Let's build a proper temperature monitor that prints a reading every 2 seconds — like a real digital thermometer.

Fill in the blanks:

In [ ]:
from machine import ADC
import math
import time

adc = ADC(26)

def read_temperature():
    """Read the thermistor and return temperature in Celsius."""
    adc_value = adc._____()                    # What method reads the ADC?
    resistance = 10000 * adc_value / (65535 - adc_value)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance/10000))
    temp_c = temp_k - 273.15
    return temp_c

reading_number = 1

while True:
    temp_c = read_temperature()
    temp_f = temp_c * _____ + _____            # Celsius to Fahrenheit formula?
    
    print("Reading #" + str(reading_number) + ":",
          round(temp_c, 1), "C /",
          round(temp_f, 1), "F")
    
    reading_number = reading_number + 1
    time.sleep(_____)                          # How many seconds between readings?

You now have a working digital thermometer! Watch it run for a while. Is the temperature stable? Does it drift?

---
## Code Along: Simple Data Logger

A thermometer shows you the temperature right now. A **data logger** records temperatures over time so you can look at the history.

We'll store readings in a Python **list** and print a summary at the end.

In [ ]:
from machine import ADC
import math
import time

adc = ADC(26)

def read_temperature():
    """Read the thermistor and return temperature in Celsius."""
    adc_value = adc.read_u16()
    resistance = 10000 * adc_value / (65535 - adc_value)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance/10000))
    temp_c = temp_k - 273.15
    return temp_c

# Collect 15 readings, one every 2 seconds (30 seconds total)
NUM_READINGS = 15
readings = []

print("Starting data log... " + str(NUM_READINGS) + " readings")
print("Try warming or cooling the sensor during the log!")
print()

for i in range(NUM_READINGS):
    temp = read_temperature()
    readings.append(round(temp, 1))
    print("Reading", i + 1, "of", NUM_READINGS, ":", round(temp, 1), "C")
    time.sleep(2)

# Print summary
print()
print("===== DATA LOG COMPLETE =====")
print("All readings:", readings)
print("Highest:", max(readings), "C")
print("Lowest:", min(readings), "C")
print("Average:", round(sum(readings) / len(readings), 1), "C")

While the data logger runs, try:
- Hold the thermistor for some readings
- Let go for some readings
- Blow on it gently

Look at your summary — did you create a temperature change that shows up in the data?

---
## Experiments

Try these modifications and observations:

1. **Body heat test:** Hold the thermistor firmly between your fingers for 30 seconds. How close to body temperature (37 C / 98.6 F) can you get it?

2. **Cooling test:** After warming it, blow on the thermistor gently. How fast does the temperature drop? Does it reach room temperature in 30 seconds?

3. **Warm object test:** Hold the thermistor near (not touching!) a warm cup of water or a laptop charger. Can you detect the heat without touching?

4. **Accuracy check:** Compare your thermometer's reading to a real thermometer or a weather app. How close is it?

---
## Challenge: Temperature Alarm

Build a **temperature alarm** — if the temperature goes above a threshold, a buzzer sounds!

You'll need to add the buzzer circuit from **Module 8** alongside your thermistor circuit.

### How it works:
1. Continuously read the temperature
2. If the temperature goes above your threshold (e.g., 30 C), sound the buzzer
3. If the temperature drops back down, turn the buzzer off
4. Print the temperature and alarm status

### Buzzer wiring reminder (Module 8):
```
   Pico GP16 ──── Buzzer (+) ──── Buzzer (-) ──── GND
```

### Useful hints:

```python
from machine import Pin, ADC, PWM

# Buzzer on GP16 using PWM
buzzer = PWM(Pin(16))
buzzer.freq(1000)          # 1000 Hz tone

# To turn buzzer ON:
buzzer.duty_u16(32768)     # 50% duty = audible tone

# To turn buzzer OFF:
buzzer.duty_u16(0)

TEMP_THRESHOLD = 30        # Degrees Celsius
```

Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Thermistor** | A resistor whose resistance changes with temperature (therm + resistor) |
| **NTC (Negative Temperature Coefficient)** | As temperature goes UP, resistance goes DOWN |
| **Voltage Divider** | Two resistors sharing voltage — the junction voltage changes when one resistor changes |
| **Steinhart-Hart Equation** | A formula that converts thermistor resistance to temperature |
| **Kelvin** | A temperature scale starting at absolute zero (0 K = -273.15 C) |
| **Data Logger** | A program that records sensor readings over time for later analysis |
| **Conversion** | Changing a value from one unit or scale to another (e.g., raw ADC to degrees) |

---
*Circuit Explorers — Module 11: Temperature Check*